## Notebook 5: Risk Segmentation & Decision Engine


The optimized threshold identified in Notebook 4 provides a decision boundary, but lending institutions rarely operate using probabilities alone.

In practice, applicants are grouped into risk categories and routed through different decision workflows. This notebook transforms calibrated default probabilities into interpretable risk bands, lending decisions, and applicant-level recommendations.

The goal is to bridge the gap between predictive modeling and actionable credit risk management.

# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib

# Load Model Artifacts and Evaluation Data

In [ ]:
x_train=pd.read_csv("x_train_processed.csv")
x_test=pd.read_csv("x_test_processed.csv")
y_train=pd.read_csv("y_train.csv").squeeze()
y_test=pd.read_csv("y_test.csv").squeeze()

best_model=joblib.load("best_baseline_model.pkl")
best_threshold=joblib.load("best_business_threshold.pkl")
preprocessor=joblib.load("preprocessor.pkl")


## Generate Default Probabilities

Generate default probability estimates for each applicant in the testing dataset.

These probabilities will serve as the foundation for risk segmentation and lending decisions.

In [ ]:
default_probabilities= best_model.predict_proba(x_test)[:,1]
default_probabilities[:10]


array([0.04729249, 0.03637318, 0.05895396, 0.12535161, 0.08177067,
       0.02158387, 0.30278365, 0.07618759, 0.24721703, 0.14755223])

In [ ]:
risk_results=pd.DataFrame({
    "Actual_Default":y_test,
    "Default_Probability":default_probabilities
})

risk_results.head()

,Actual_Default,Default_Probability
0,0,0.047292
1,0,0.036373
2,0,0.058954
3,0,0.125352
4,0,0.081771


## Risk Segmentation Framework

Raw probability values can be difficult for business stakeholders to interpret.

To improve usability, applicants are grouped into risk bands based on predicted default probability.

The segmentation strategy is anchored around the business-optimal threshold identified in Notebook 4.

In [ ]:
def assign_risk_band(probability):
  if probability<= 0.05:
    return "Low Risk"
  elif probability < best_threshold :
    return "Moderate Risk"
  elif probability < 0.30:
    return "High Risk"
  else:
    return "Critical Risk"


## Validate Risk Segmentation

Measure the observed default rate within each risk band to determine whether the segmentation framework reflects actual borrower risk.

In [ ]:
risk_results["Risk_Band"] = risk_results["Default_Probability"].apply(assign_risk_band)
risk_results.head()

,Actual_Default,Default_Probability,Risk_Band
0,0,0.047292,Low Risk
1,0,0.036373,Low Risk
2,0,0.058954,Moderate Risk
3,0,0.125352,Moderate Risk
4,0,0.081771,Moderate Risk


## Lending Decision Framework

Risk bands are translated into lending actions.

The objective is to create a decision engine that reflects practical credit risk workflows rather than relying solely on probability scores.

In [ ]:
def assign_decision(risk_band):

  if risk_band=="Low Risk":
    return "Approve"
  elif risk_band in ["Moderate Risk","High Risk"]:
    return "Manual Review"
  else:
    return "Reject"

In [ ]:
risk_results["Decision"]= risk_results["Risk_Band"].apply(assign_decision)

risk_results.head()

,Actual_Default,Default_Probability,Risk_Band,Decision
0,0,0.047292,Low Risk,Approve
1,0,0.036373,Low Risk,Approve
2,0,0.058954,Moderate Risk,Manual Review
3,0,0.125352,Moderate Risk,Manual Review
4,0,0.081771,Moderate Risk,Manual Review


In [ ]:
risk_results["Decision"].value_counts()


,count
Decision,
Manual Review,36105
Approve,11912
Reject,3053


In [ ]:
risk_results["Risk_Band"].value_counts()

,count
Risk_Band,
Moderate Risk,26821
Low Risk,11912
High Risk,9284
Critical Risk,3053


In [ ]:
risk_results.groupby("Risk_Band")["Actual_Default"].mean().sort_values()


,Actual_Default
Risk_Band,
Low Risk,0.027787
Moderate Risk,0.083740
High Risk,0.220056
Critical Risk,0.429414


### Finding: Default Rates Increase Consistently Across Risk Bands

Observed default rates increase steadily from Low Risk to Critical Risk.

Low Risk: ~2.8%

Moderate Risk: ~8.4%

High Risk: ~22.0%

Critical Risk: ~42.9%


### Why This Matters:

Applicants classified as Critical Risk are approximately 15 times more likely to default than applicants classified as Low Risk.

This demonstrates that the segmentation framework is not arbitrary and captures meaningful differences in borrower risk.

## Validate Decision Outcomes

Evaluate whether the decision framework produces meaningful separation in observed default rates.

In [ ]:
risk_results.groupby("Decision")["Actual_Default"].mean().sort_values()

,Actual_Default
Decision,
Approve,0.027787
Manual Review,0.118792
Reject,0.429414


### Finding: Decision Categories Reflect Meaningful Risk Differences

Observed default rates increase substantially across decision outcomes.

Approve: ~2.7%

Manual Review: ~12.0%

Reject: ~42.9%


### Business Interpretation

Applicants routed to Reject are approximately twice as likely to default as applicants routed to Manual Review and more than six times as likely to default as approved applicants.

This indicates that the decision engine successfully prioritizes higher-risk borrowers for stricter intervention.

## Applicant Recommendation Engine

Generate human-readable recommendations that explain the appropriate next action for each applicant.

These recommendations are intended to support credit analysts and lending teams.

In [ ]:
def generate_recommendation(risk_band,decision):
  if decision =="Approve":
    return "Applicant falls within an acceptable risk range and can be considered for approval."

  elif decision=="Manual Review":
    if risk_band=="Moderate Risk":
      return "Applicant shows moderate risk. Request additional income or employment verification."
    else:
      return "Applicant shows elevated risk. Perform enhanced credit review before approval."

  else:
    return "Application exceeds the business risk threshold and should be rejected or escalated."


In [ ]:
risk_results["Recommendation"]=risk_results.apply(
    lambda row: generate_recommendation(
        row["Risk_Band"],
        row["Decision"]
    ),
    axis=1
)

risk_results.head()

,Actual_Default,Default_Probability,Risk_Band,Decision,Recommendation
0,0,0.047292,Low Risk,Approve,Applicant falls within an acceptable risk rang...
1,0,0.036373,Low Risk,Approve,Applicant falls within an acceptable risk rang...
2,0,0.058954,Moderate Risk,Manual Review,Applicant shows moderate risk. Request additio...
3,0,0.125352,Moderate Risk,Manual Review,Applicant shows moderate risk. Request additio...
4,0,0.081771,Moderate Risk,Manual Review,Applicant shows moderate risk. Request additio...


In [ ]:
applicant_index= 0

applicant_report= risk_results.iloc[applicant_index]
applicant_report


,0
Actual_Default,0
Default_Probability,0.047292
Risk_Band,Low Risk
Decision,Approve
Recommendation,Applicant falls within an acceptable risk rang...


## Applicant-Level Reporting

Create applicant-specific reports containing:

- Default Probability
- Risk Band
- Lending Decision
- Recommendation

This output represents the final business-facing layer of RiskLens.

In [ ]:
def generate_applicant_report(index):
  applicant = risk_results.iloc[index]

  report={
      "Applicant_Index":index,
      "Default_Probability":round(applicant["Default_Probability"],4),
      "Risk_Band":applicant["Risk_Band"],
      "Decision":applicant["Decision"],
      "Recommendation":applicant["Recommendation"],
      "Actual_Default":applicant["Actual_Default"]
  }

  return report



In [ ]:
generate_applicant_report(30)

{'Applicant_Index': 30,
 'Default_Probability': np.float64(0.1368),
 'Risk_Band': 'Moderate Risk',
 'Decision': 'Manual Review',
 'Recommendation': 'Applicant shows moderate risk. Request additional income or employment verification.',
 'Actual_Default': np.int64(1)}

## Simulated Applicant Assessment

Evaluate a previously unseen applicant and generate a complete RiskLens decision report.

This demonstrates how the system would operate in a real-world lending environment.

In [ ]:
applicant = pd.DataFrame([{
    "Age": 32,
    "Income": 75000,
    "LoanAmount": 120000,
    "CreditScore": 690,
    "MonthsEmployed": 48,
    "NumCreditLines": 2,
    "InterestRate": 11.5,
    "LoanTerm": 36,
    "DTIRatio": 0.35,
    "EmploymentType": "Full-time",
    "HasMortgage": "No",
    "HasDependents": "Yes",
    "LoanPurpose": "Home",
    "HasCoSigner": "No",
    "MaritalStatus": "Married",
    "Education": "PhD"
}])

applicant

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,EmploymentType,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,MaritalStatus,Education
0,32,75000,120000,690,48,2,11.5,36,0.35,Full-time,No,Yes,Home,No,Married,PhD


In [ ]:
applicant_processed= preprocessor.transform(applicant)
applicant_processed

array([[-7.67762579e-01, -1.93233816e-01, -1.06156358e-01,
         7.27602940e-01, -3.34009813e-01, -4.49335224e-01,
        -2.99503424e-01,  5.19126128e-04, -6.51523000e-01,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  1.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         0.00000000e+00]])

In [ ]:
applicant_probability = best_model.predict_proba(applicant_processed)[:,1][0]
applicant_risk_band = assign_risk_band(applicant_probability)
applicant_decision = assign_decision(applicant_risk_band)
applicant_recommendation = generate_recommendation(
    applicant_risk_band,
    applicant_decision
)

applicant_report = {
    "Default_Probability": round(applicant_probability, 4),
    "Risk_Band": applicant_risk_band,
    "Decision": applicant_decision,
    "Recommendation": applicant_recommendation
}

applicant_report

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but GradientBoostingClassifier was fitted with feature names
  warnings.warn(


{'Default_Probability': np.float64(0.0633),
 'Risk_Band': 'Moderate Risk',
 'Decision': 'Manual Review',
 'Recommendation': 'Applicant shows moderate risk. Request additional income or employment verification.'}

The applicant-level assessment demonstrates how RiskLens can be integrated into a lending workflow, transforming raw applicant information into a probability estimate, risk classification, lending decision, and actionable recommendation.

In [ ]:
risk_results.to_csv("risklens_decision_results.csv",index=False)

## Key Findings

- Predicted probabilities were successfully transformed into interpretable risk categories.
- Risk bands demonstrated steadily increasing observed default rates, validating the segmentation framework.
- The decision engine successfully differentiated between lower-risk and higher-risk applicants.
- Applicant recommendations were generated to support practical lending workflows.
- RiskLens can now evaluate individual applicants and produce business-oriented risk assessments.

The project has evolved from a predictive model into a complete credit risk decisioning system capable of generating actionable lending recommendations.